In [0]:
dbutils.widgets.text("environment", "dev")

dbutils.widgets.dropdown(
    "dataset_name",
    "products",
    [
        "products",
        "customers",
        "stores",
        "promotions",
        "inventory"
    ]
)

dbutils.widgets.text("pipeline_run_id", "")
dbutils.widgets.text("catalog_name", "retail_lakehouse")

In [0]:
import uuid

environment = dbutils.widgets.get("environment").strip()
dataset_name = dbutils.widgets.get("dataset_name").strip().lower()
pipeline_run_id = dbutils.widgets.get("pipeline_run_id").strip()
catalog_name = dbutils.widgets.get("catalog_name").strip()

if not pipeline_run_id:
    pipeline_run_id = str(uuid.uuid4())

print(f"Environment     : {environment}")
print(f"Dataset         : {dataset_name}")
print(f"Pipeline run ID : {pipeline_run_id}")
print(f"Catalog         : {catalog_name}")

In [0]:
for variable_name in [
    "transformed_df",
    "validated_df",
    "valid_df",
    "rejected_df",
    "deduplicated_df",
    "silver_output_df"
]:
    globals().pop(variable_name, None)

In [0]:
bronze_table = f"{catalog_name}.bronze.{dataset_name}"
silver_table = f"{catalog_name}.silver.{dataset_name}"

rejected_table = (
    f"{catalog_name}.quarantine."
    f"{dataset_name}_business_rejected"
)

audit_table = f"{catalog_name}.audit.pipeline_runs"

silver_control_table = (
    f"{catalog_name}.audit.silver_reference_processed_batches"
)

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_control_table}
(
    dataset_name STRING,
    source_table STRING,
    target_table STRING,
    source_max_ingestion_timestamp TIMESTAMP,
    silver_pipeline_run_id STRING,
    records_read BIGINT,
    records_written BIGINT,
    records_rejected BIGINT,
    processed_timestamp TIMESTAMP,
    processing_status STRING
)
USING DELTA
""")

In [0]:
from pyspark.sql import functions as F

last_processed_timestamp = (
    spark.table(silver_control_table)
    .filter(
        (F.col("dataset_name") == dataset_name)
        & (F.col("processing_status") == "SUCCESS")
    )
    .agg(
        F.max("source_max_ingestion_timestamp")
        .alias("last_processed_timestamp")
    )
    .first()["last_processed_timestamp"]
)

print(f"Last processed timestamp: {last_processed_timestamp}")

In [0]:
bronze_df = spark.table(bronze_table)

if last_processed_timestamp is not None:
    bronze_df = bronze_df.filter(
        F.col("_ingestion_timestamp")
        > F.lit(last_processed_timestamp)
    )

In [0]:
records_read = bronze_df.count()

if records_read == 0:
    dbutils.notebook.exit(
        f"No new Bronze records for dataset={dataset_name}"
    )

source_max_ingestion_timestamp = (
    bronze_df
    .agg(F.max("_ingestion_timestamp").alias("max_ts"))
    .first()["max_ts"]
)

print(f"Records read: {records_read}")

In [0]:
if dataset_name == "products":
    transformed_df = (
        bronze_df
        .withColumn(
            "product_id",
            F.upper(F.trim("product_id"))
        )
        .withColumn(
            "product_name",
            F.initcap(F.trim("product_name"))
        )
        .withColumn(
            "category",
            F.initcap(F.trim("category"))
        )
        .withColumn(
            "brand",
            F.initcap(F.trim("brand"))
        )
        .withColumn(
            "standard_price",
            F.col("standard_price").cast("decimal(12,2)")
        )
        .withColumn(
            "cost_price",
            F.col("cost_price").cast("decimal(12,2)")
        )
        .withColumn(
            "product_status",
            F.upper(F.trim("product_status"))
        )
    )

In [0]:
if dataset_name == "products":
    validated_df = (
        transformed_df
        .withColumn(
            "rejection_reasons",
            F.array_compact(
                F.array(
                    F.when(
                        F.col("product_id").isNull(),
                        "INVALID_PRODUCT_ID"
                    ),
                    F.when(
                        F.col("product_name").isNull(),
                        "INVALID_PRODUCT_NAME"
                    ),
                    F.when(
                        F.col("standard_price").isNull()
                        | (F.col("standard_price") < 0),
                        "INVALID_STANDARD_PRICE"
                    ),
                    F.when(
                        F.col("cost_price").isNull()
                        | (F.col("cost_price") < 0),
                        "INVALID_COST_PRICE"
                    ),
                    F.when(
                        ~F.col("product_status").isin(
                            "ACTIVE",
                            "INACTIVE"
                        ),
                        "INVALID_PRODUCT_STATUS"
                    )
                )
            )
        )
    )

In [0]:
#Transforming Products:

if dataset_name == "products":
    transformed_df = (
        bronze_df
        .withColumn(
            "product_id",
            F.upper(F.trim("product_id"))
        )
        .withColumn(
            "product_name",
            F.initcap(F.trim("product_name"))
        )
        .withColumn(
            "category",
            F.initcap(F.trim("category"))
        )
        .withColumn(
            "brand",
            F.initcap(F.trim("brand"))
        )
        .withColumn(
            "standard_price",
            F.col("standard_price").cast("decimal(12,2)")
        )
        .withColumn(
            "cost_price",
            F.col("cost_price").cast("decimal(12,2)")
        )
        .withColumn(
            "product_status",
            F.upper(F.trim("product_status"))
        )
    )

    #Validating

    if dataset_name == "products":
        validated_df = (
        transformed_df
        .withColumn(
            "rejection_reasons",
            F.array_compact(
                F.array(
                    F.when(
                        F.col("product_id").isNull(),
                        "INVALID_PRODUCT_ID"
                    ),
                    F.when(
                        F.col("product_name").isNull(),
                        "INVALID_PRODUCT_NAME"
                    ),
                    F.when(
                        F.col("standard_price").isNull()
                        | (F.col("standard_price") < 0),
                        "INVALID_STANDARD_PRICE"
                    ),
                    F.when(
                        F.col("cost_price").isNull()
                        | (F.col("cost_price") < 0),
                        "INVALID_COST_PRICE"
                    ),
                    F.when(
                        ~F.col("product_status").isin(
                            "ACTIVE",
                            "INACTIVE"
                        ),
                        "INVALID_PRODUCT_STATUS"
                    )
                )
            )
        )
    )

In [0]:
#Transforming Customers:

if dataset_name == "customers":
    transformed_df = (
        bronze_df
        .withColumn(
            "customer_id",
            F.upper(F.trim("customer_id"))
        )
        .withColumn(
            "customer_name",
            F.initcap(F.trim("customer_name"))
        )
        .withColumn(
            "age",
            F.col("age").cast("integer")
        )
        .withColumn(
            "city",
            F.initcap(F.trim("city"))
        )
        .withColumn(
            "state",
            F.initcap(F.trim("state"))
        )
        .withColumn(
            "region",
            F.initcap(F.trim("region"))
        )
        .withColumn(
            "loyalty_tier",
            F.initcap(F.trim("loyalty_tier"))
        )
        .withColumn(
            "registration_date",
            F.to_date("registration_date")
        )
        .withColumn(
            "customer_age_group",
            F.when(F.col("age") < 25, "18-24")
            .when(F.col("age") < 35, "25-34")
            .when(F.col("age") < 45, "35-44")
            .when(F.col("age") < 60, "45-59")
            .otherwise("60+")
        )
    )

    #Validating

    if dataset_name == "customers":
        validated_df = (
        transformed_df
        .withColumn(
            "rejection_reasons",
            F.array_compact(
                F.array(
                    F.when(
                        F.col("customer_id").isNull(),
                        "INVALID_CUSTOMER_ID"
                    ),
                    F.when(
                        F.col("age").isNull()
                        | (F.col("age") < 18)
                        | (F.col("age") > 100),
                        "INVALID_AGE"
                    ),
                    F.when(
                        ~F.col("loyalty_tier").isin(
                            "Bronze",
                            "Silver",
                            "Gold",
                            "Platinum"
                        ),
                        "INVALID_LOYALTY_TIER"
                    ),
                    F.when(
                        F.col("registration_date").isNull(),
                        "INVALID_REGISTRATION_DATE"
                    )
                )
            )
        )
    )

In [0]:
#Transforming Store

if dataset_name == "stores":
    transformed_df = (
        bronze_df
        .withColumn(
            "store_id",
            F.upper(F.trim("store_id"))
        )
        .withColumn(
            "store_name",
            F.initcap(F.trim("store_name"))
        )
        .withColumn(
            "city",
            F.initcap(F.trim("city"))
        )
        .withColumn(
            "state",
            F.initcap(F.trim("state"))
        )
        .withColumn(
            "region",
            F.initcap(F.trim("region"))
        )
        .withColumn(
            "store_size",
            F.initcap(F.trim("store_size"))
        )
    )

    #Validating

    if dataset_name == "stores":
        validated_df = (
        transformed_df
        .withColumn(
            "rejection_reasons",
            F.array_compact(
                F.array(
                    F.when(
                        F.col("store_id").isNull(),
                        "INVALID_STORE_ID"
                    ),
                    F.when(
                        F.col("store_name").isNull(),
                        "INVALID_STORE_NAME"
                    ),
                    F.when(
                        ~F.col("store_size").isin(
                            "Small",
                            "Medium",
                            "Large"
                        ),
                        "INVALID_STORE_SIZE"
                    )
                )
            )
        )
    )

In [0]:
#Transforming Promotions;

if dataset_name == "promotions":
    transformed_df = (
        bronze_df
        .withColumn(
            "promotion_id",
            F.upper(F.trim("promotion_id"))
        )
        .withColumn(
            "promotion_name",
            F.initcap(F.trim("promotion_name"))
        )
        .withColumn(
            "category",
            F.initcap(F.trim("category"))
        )
        .withColumn(
            "discount_type",
            F.upper(F.trim("discount_type"))
        )
        .withColumn(
            "discount_value",
            F.col("discount_value").cast("decimal(12,2)")
        )
        .withColumn(
            "start_date",
            F.to_date("start_date")
        )
        .withColumn(
            "end_date",
            F.to_date("end_date")
        )
        .withColumn(
            "promotion_status",
            F.when(
                F.current_date().between(
                    F.col("start_date"),
                    F.col("end_date")
                ),
                "ACTIVE"
            ).otherwise("INACTIVE")
        )
    )

    #Validating

    if dataset_name == "promotions":
        validated_df = (
        transformed_df
        .withColumn(
            "rejection_reasons",
            F.array_compact(
                F.array(
                    F.when(
                        F.col("promotion_id").isNull(),
                        "INVALID_PROMOTION_ID"
                    ),
                    F.when(
                        F.col("promotion_name").isNull(),
                        "INVALID_PROMOTION_NAME"
                    ),
                    F.when(
                        F.col("discount_value").isNull()
                        | (F.col("discount_value") < 0),
                        "INVALID_DISCOUNT_VALUE"
                    ),
                    F.when(
                        ~F.col("discount_type").isin(
                            "PERCENTAGE",
                            "FIXED_AMOUNT"
                        ),
                        "INVALID_DISCOUNT_TYPE"
                    ),
                    F.when(
                        F.col("start_date").isNull(),
                        "INVALID_START_DATE"
                    ),
                    F.when(
                        F.col("end_date").isNull(),
                        "INVALID_END_DATE"
                    ),
                    F.when(
                        F.col("end_date") < F.col("start_date"),
                        "END_DATE_BEFORE_START_DATE"
                    )
                )
            )
        )
    )

In [0]:
#Transforming Inventory

if dataset_name == "inventory":
    transformed_df = (
        bronze_df
        .withColumn(
            "store_id",
            F.upper(F.trim("store_id"))
        )
        .withColumn(
            "product_id",
            F.upper(F.trim("product_id"))
        )
        .withColumn(
            "inventory_date",
            F.to_date("inventory_date")
        )
        .withColumn(
            "opening_stock",
            F.col("opening_stock").cast("integer")
        )
        .withColumn(
            "received_quantity",
            F.col("received_quantity").cast("integer")
        )
        .withColumn(
            "sold_quantity",
            F.col("sold_quantity").cast("integer")
        )
        .withColumn(
            "damaged_quantity",
            F.col("damaged_quantity").cast("integer")
        )
        .withColumn(
            "closing_stock",
            F.col("closing_stock").cast("integer")
        )
        .withColumn(
            "calculated_closing_stock",
            F.col("opening_stock")
            + F.col("received_quantity")
            - F.col("sold_quantity")
            - F.col("damaged_quantity")
        )
    )

    #Validating

    if dataset_name == "inventory":
        validated_df = (
        transformed_df
        .withColumn(
            "rejection_reasons",
            F.array_compact(
                F.array(
                    F.when(
                        F.col("store_id").isNull(),
                        "INVALID_STORE_ID"
                    ),
                    F.when(
                        F.col("product_id").isNull(),
                        "INVALID_PRODUCT_ID"
                    ),
                    F.when(
                        F.col("inventory_date").isNull(),
                        "INVALID_INVENTORY_DATE"
                    ),
                    F.when(
                        F.col("opening_stock").isNull()
                        | (F.col("opening_stock") < 0),
                        "INVALID_OPENING_STOCK"
                    ),
                    F.when(
                        F.col("received_quantity").isNull()
                        | (F.col("received_quantity") < 0),
                        "INVALID_RECEIVED_QUANTITY"
                    ),
                    F.when(
                        F.col("sold_quantity").isNull()
                        | (F.col("sold_quantity") < 0),
                        "INVALID_SOLD_QUANTITY"
                    ),
                    F.when(
                        F.col("damaged_quantity").isNull()
                        | (F.col("damaged_quantity") < 0),
                        "INVALID_DAMAGED_QUANTITY"
                    ),
                    F.when(
                        F.col("closing_stock").isNull()
                        | (F.col("closing_stock") < 0),
                        "INVALID_CLOSING_STOCK"
                    ),
                    F.when(
                        F.col("closing_stock") != F.col("calculated_closing_stock"),
                        "STOCK_MISMATCH"
                    )
                )
            )
        )
    )

In [0]:
if "transformed_df" not in globals():
    raise RuntimeError(
        f"transformed_df was not created for dataset={dataset_name}"
    )

if "validated_df" not in globals():
    raise RuntimeError(
        f"validated_df was not created for dataset={dataset_name}"
    )

In [0]:
validated_df = validated_df.withColumn(
    "data_quality_status",
    F.when(
        F.size("rejection_reasons") == 0,
        "VALID"
    ).otherwise("REJECTED")
)

valid_df = validated_df.filter(
    F.col("data_quality_status") == "VALID"
)

rejected_df = validated_df.filter(
    F.col("data_quality_status") == "REJECTED"
)

records_rejected = rejected_df.count()
valid_count = valid_df.count()

print(f"Records read     : {records_read}")
print(f"Valid records    : {valid_count}")
print(f"Rejected records : {records_rejected}")

In [0]:
#Split valid and rejected data
if records_rejected > 0:
    rejected_output_df = (
        rejected_df
        .withColumn(
            "_silver_pipeline_run_id",
            F.lit(pipeline_run_id)
        )
        .withColumn(
            "_rejected_timestamp",
            F.current_timestamp()
        )
    )

    (
        rejected_output_df
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(rejected_table)
    )

In [0]:
#Deduplication
business_keys = {
    "products": ["product_id"],
    "customers": ["customer_id"],
    "stores": ["store_id"],
    "promotions": ["promotion_id"],
    "inventory": [
        "store_id",
        "product_id",
        "inventory_date"
    ]
}

In [0]:
from pyspark.sql.window import Window

key_columns = business_keys[dataset_name]

dedup_window = (
    Window
    .partitionBy(*key_columns)
    .orderBy(
        F.col("_ingestion_timestamp").desc(),
        F.col("_record_hash").desc()
    )
)

deduplicated_df = (
    valid_df
    .withColumn(
        "_row_number",
        F.row_number().over(dedup_window)
    )
    .filter(F.col("_row_number") == 1)
    .drop("_row_number")
)

In [0]:
#Adding Silver Metadata
silver_output_df = (
    deduplicated_df
    .withColumn(
        "_silver_pipeline_run_id",
        F.lit(pipeline_run_id)
    )
    .withColumn(
        "_silver_updated_timestamp",
        F.current_timestamp()
    )
)

In [0]:
#Dropping Bronze-only technical columns
columns_to_drop = [
    "_corrupt_record",
    "rejection_reasons",
    "data_quality_status"
]

silver_output_df = silver_output_df.drop(*columns_to_drop)

In [0]:
#Delta Merge:

from delta.tables import DeltaTable

merge_condition = " AND ".join(
    [
        f"target.{column_name} = source.{column_name}"
        for column_name in key_columns
    ]
)

print(merge_condition)

if not spark.catalog.tableExists(silver_table):
    (
        silver_output_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(silver_table)
    )

    print(f"Created Silver table: {silver_table}")

else:
    target_delta = DeltaTable.forName(
        spark,
        silver_table
    )

    (
        target_delta.alias("target")
        .merge(
            silver_output_df.alias("source"),
            merge_condition
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(f"Merged into Silver table: {silver_table}")

In [0]:
records_written = silver_output_df.count()

In [0]:
#Write audit before control completion

from datetime import datetime

audit_schema = spark.table(audit_table).schema

silver_audit_data = [{
    "pipeline_run_id": pipeline_run_id,
    "pipeline_name": "retail_reference_silver_transformation",
    "notebook_name": "02_transform_reference_silver",
    "source_system": f"bronze_{dataset_name}",
    "source_file_name": None,
    "source_file_path": bronze_table,
    "layer": "silver",
    "load_type": "INCREMENTAL",
    "run_status": "SUCCESS",
    "records_read": int(records_read),
    "records_written": int(records_written),
    "records_rejected": int(records_rejected),
    "start_timestamp": datetime.now(),
    "end_timestamp": datetime.now(),
    "error_message": None,
    "created_timestamp": datetime.now()
}]

audit_df = spark.createDataFrame(
    silver_audit_data,
    schema=audit_schema
)

(
    audit_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(audit_table)
)

In [0]:
#Register the processed watermark last

control_schema = spark.table(silver_control_table).schema

control_data = [{
    "dataset_name": dataset_name,
    "source_table": bronze_table,
    "target_table": silver_table,
    "source_max_ingestion_timestamp":
        source_max_ingestion_timestamp,
    "silver_pipeline_run_id": pipeline_run_id,
    "records_read": int(records_read),
    "records_written": int(records_written),
    "records_rejected": int(records_rejected),
    "processed_timestamp": datetime.now(),
    "processing_status": "SUCCESS"
}]

control_df = spark.createDataFrame(
    control_data,
    schema=control_schema
)

(
    control_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(silver_control_table)
)

In [0]:
for table_name in [
    "products",
    "customers",
    "stores",
    "promotions",
    "inventory"
]:
    full_name = f"{catalog_name}.silver.{table_name}"

    if spark.catalog.tableExists(full_name):
        print(
            full_name,
            spark.table(full_name).count()
        )
    else:
        print(f"{full_name} (not created yet)")